In [1]:
import pandas as pd
import os
import librosa as librs
import IPython.display as ipd
import matplotlib.pyplot as plt
import numpy as np
import pywt
import time
from pydub import AudioSegment
import soundfile as sf

In [ ]:
# Calcolo della trasformata wavelet
file_path = './archive/fold16/input/0.wav'
y, sr = librs.load(file_path)
# Set the start and end points for trimming
start_time = int(0.15 * sr)
end_time = int(len(y) - 0.15 * sr)
# Trim the audio
yc = y[start_time:end_time]

wavelet = 'cmor'  # Tipo di wavelet (può essere cambiato)
coefficients, frequencies = pywt.cwt(yc, np.arange(1, 128), wavelet)

# Visualizzazione del segnale audio e della trasformata wavelet
plt.figure(figsize=(10, 5))
plt.subplot(2, 1, 1)
librs.display.waveshow(yc, sr=sr)
plt.ylabel('Width')
plt.title('Audio signal')

plt.subplots_adjust(hspace=0.5)

plt.subplot(2, 1, 2)
plt.imshow(np.abs(coefficients), extent=[0, len(yc), 1, 128], cmap='jet', aspect='auto')
plt.colorbar(label='Coefficient')
plt.xlabel('Time')
plt.ylabel('Frequency')
plt.title('Wavelet transorm')
plt.show()

C:\Users\massi\anaconda3\envs\key\lib\site-packages\pywt\_cwt.py:117: FutureWarning: Wavelets from the family cmor, without parameters specified in the name are deprecated. The name should takethe form cmorB-C where B and C are floats representing the bandwidth frequency and center frequency, respectively (example: cmor1.5-1.0).
  wavelet = DiscreteContinuousWavelet(wavelet)


In [ ]:
#tecnica di riconoscimento picchi 1: calcolo del rumore e poi considerare picco ciò che è x volte superiore
output_dir = "archive/fold16/output_directory"  # Replace with the path to your output directory
click_time = 0.5
#peak_times = []

def split_trim(input_file, peak_times, click_time, audio, sr):
    #audiot = AudioSegment.from_wav(file_path)
    #print(len(peak_times)-1)
    for t in range(len(peak_times)):
        print('########################################################')
        start_time = int((peak_times[t] - (click_time/2)) * sr)
        end_time = int((peak_times[t] + (click_time/2)) * sr)
        #end_time = int(len(audiot) - (peak_times[i] + (click_time/2)) * sr)
        split_audio = audio[start_time:end_time]
        file_name_without_extension = os.path.splitext(os.path.basename(input_file))[0]
        #print(file_name_without_extension)
        output_filename = os.path.join(output_dir, f"{file_name_without_extension}_split_{t+1}.wav")
        #split_audio.export(output_filename, format="wav")
        sf.write(output_filename, split_audio, sr)
        print(f"Split {t+1} saved as {output_filename}")                       

def recognize_peak(file_path, y, sr):
    peak_times = []
    window_noise_calculation = 0.15
    peak_recognize_scale = 3
    #click_time = 0.6
    # Calcolo della trasformata wavelet
    waveletname = 'cmor'  # Tipo di wavelet (può essere cambiato)
    scales = np.arange(1, 128)
    coefficients, frequencies = pywt.cwt(y, scales, waveletname)
    # Calcolo del "threshold" del rumore di fondo
    time_window = int(window_noise_calculation * sr)  # x secondi dai quali prendere il rumore di fondo
    noise_level = np.mean(np.abs(coefficients[:, :time_window]))
    noise_level = (noise_level+0.002)/2
    # Impostazione del "threshold" al x del rumore di fondo
    threshold = peak_recognize_scale * noise_level
    # Analisi della traccia e memorizzazione dei momenti in cui vengono trovati i picchi
    i = 0
    peak_count = 0
    maybe_peak = 0
    while i < len(coefficients[0]):
        peak_found = False
        for j in range(len(coefficients)):
            maybe_peak = abs(coefficients[j, i] - coefficients[j, i - 1])
            if i > 0 and maybe_peak > threshold :
                time_of_peak = librs.samples_to_time(i, sr=sr)  # Converte i campioni in tempo
                peak_times.append(time_of_peak)
                i += int(click_time * sr)  # Salta in avanti di x secondi
                peak_found = True
                peak_count+=1
                break
        if not peak_found:
            i += 1
    
    # Rappresentazione grafica dei picchi trovati sull'onda audio
    plt.figure(figsize=(10, 6))
    plt.plot(np.arange(len(y)) / sr, y, label='Audio wave')  # Grafico dell'onda audio
    # Indicazione dei picchi trovati sull'onda audio
    for peak_time in peak_times:
        plt.axvline(x=peak_time, color='red', linestyle='--', alpha=0.7)
    #split_trim(peak_times, click_time, y, sr)
    plt.xlabel('Time (seconds)')
    plt.ylabel('Width')
    plt.title('Spikes on audio wave')
    plt.legend()
    plt.show()
    # Stampa del numero di picchi sopra il "threshold" e del rumore di fondo
    print(f"Valore del rumore di fondo: {noise_level}")
    print(f"Numero di picchi sopra il threshold: {peak_count}")
    split_trim(file_path, peak_times, click_time, y, sr)

In [ ]:
input_directory = "archive/fold16/input"  # Replace with the path to your input directory

for filename in os.listdir(input_directory):
    if filename.endswith(".wav"):
        input_file = os.path.join(input_directory, filename)
        audio, sr = librs.load(input_file)
        # Calculate the time range to exclude the first and the last 0.1 seconds
        start_time = int((click_time/4) * sr)
        end_time = int(len(audio) - (click_time/4) * sr)
        # Trim the audio
        audio_cleared = audio[start_time:end_time]
        recognize_peak(input_file, audio_cleared, sr)
        #split_trim(input_file, peak_times, click_time, audio_cleared, sr)